# 07 — DeepSpeed ZeRO-3 Experiment

Shard optimizer states, gradients, **and** model parameters across GPUs.
Optionally offload to CPU. The industry standard for models that don't
fit on one device.

**ZeRO-3 must be launched with the `deepspeed` CLI**, not plain `python`.

In [ ]:
import sys
sys.path.append("..")

import torch
try:
    import deepspeed
    print('DeepSpeed:', deepspeed.__version__)
except ImportError:
    print('DeepSpeed not installed. Run: pip install deepspeed mpi4py')

print(f'GPUs available: {torch.cuda.device_count()}')

In [ ]:
from src.llm_optimization.core import load_config
config = load_config('./configs/zero3.yaml')
print('Stage:', config.zero3.zero_stage)
print('Offload optimizer:', config.zero3.offload_optimizer)
print('Offload params:', config.zero3.offload_param)

In [ ]:
import json
from src.llm_optimization.training import build_zero3_config

ds_config = build_zero3_config(
    config.zero3,
    batch_size=config.training.per_device_train_batch_size,
    grad_accum=config.training.gradient_accumulation_steps,
)
print(json.dumps(ds_config, indent=2))

In [ ]:
from transformers import AutoTokenizer
from src.llm_optimization.data import load_and_prepare_data, QADataset

train_df, _, val_df = load_and_prepare_data(config.data)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, config.data.max_length)
val_ds = QADataset(val_df, tokenizer, config.data.max_length)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
from src.llm_optimization.training import build_zero3_trainer

trainer, model, ds_path = build_zero3_trainer(
    config, train_ds, val_ds, tokenizer
)
print('Trainer ready.')
print('DeepSpeed config:', ds_path)

## Launch via the `deepspeed` CLI

```bash
# Single GPU with CPU offload
deepspeed --num_gpus=1 scripts/train_zero3.py --config configs/zero3.yaml

# Multi-GPU
deepspeed --num_gpus=4 scripts/train_zero3.py --config configs/zero3.yaml
```

In [ ]:
n_gpus = max(1, torch.cuda.device_count())
cmd = ['deepspeed', f'--num_gpus={n_gpus}',
       'scripts/train_zero3.py', '--config', 'configs/zero3.yaml']
print('Command to run in terminal:')
print('  ' + ' '.join(cmd))
print()
print('To execute: uncomment the line below')
print('# import subprocess; subprocess.run(cmd, check=True)')

## ZeRO stage comparison

| Stage | Shards | Speed |
|-------|--------|-------|
| ZeRO-1 | Optimizer | Fastest |
| ZeRO-2 | + Gradients | Fast |
| ZeRO-3 | + Parameters | Slower |
| ZeRO-3 + Offload | + CPU RAM | Slowest but smallest |

**When to use:** only when the model doesn't fit with Stage 1 or 2.